# S5 — Dataset-driven pressing analysis

Every ranking, title, and team characteristic is derived from the teams present in the user-provided data. Anchors belong to the team in possession; S5 attributes the pressure to that team's opponent before aggregating results.


In [ ]:
from pathlib import Path
import os, sys, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
OUTPUT_DIR = Path(os.environ['OUTPUT_DIR'])
DATASET_NAME = os.environ.get('DATASET_NAME', 'User dataset')
features = pd.read_parquet(OUTPUT_DIR / 'spn_features.parquet')


In [ ]:
from src.spn_model import score_bundle
from src.press_analysis import attach_sequence_change, style_fingerprint, team_characteristics, team_summary

scored = attach_sequence_change(score_bundle(OUTPUT_DIR / 'spn_model_bundle.joblib', features))
summary = team_summary(scored, min_sequences=5)
display(summary)


## 1. Data coverage


In [ ]:
summary.sort_values('n_sequences').plot.barh(x='team_name', y='n_sequences', legend=False, title=f'{DATASET_NAME}: high-press sequences by team')
plt.xlabel('Sequences'); plt.show()


## 2. Pressing intensity


In [ ]:
summary.sort_values('mean_pressure').plot.barh(x='team_name', y='mean_pressure', legend=False, title=f'{DATASET_NAME}: mean carrier pressure')
plt.xlabel('Mean P_total'); plt.show()


## 3. Pressing efficiency ranking


In [ ]:
ranking = summary[summary['eligible']].sort_values('press_efficiency', ascending=False)
display(ranking[['team_name','n_sequences','press_efficiency','mean_success_probability']])
ranking.plot.barh(x='team_name', y='press_efficiency', legend=False, title=f'{DATASET_NAME}: pressing efficiency')
plt.show()


## 4. Intensity versus efficiency


In [ ]:
ax = summary.plot.scatter(x='mean_pressure', y='press_efficiency', title=f'{DATASET_NAME}: intensity and efficiency')
for _, row in summary.iterrows(): ax.annotate(row['team_name'], (row['mean_pressure'], row['press_efficiency']))
plt.show()


## 5. Team style fingerprint


In [ ]:
fingerprint = style_fingerprint(summary)
display(fingerprint)
fingerprint.set_index('team_name').filter(like='z_').T.plot(marker='o', title=f'{DATASET_NAME}: relative team styles')
plt.show()


## 6. Data-relative team characteristics


In [ ]:
display(team_characteristics(summary).pivot(index='team_name', columns='dimension', values='level'))
